In [16]:
import import_ipynb
import numpy as np
from sn_curve import cycles_to_failure


def miner_damage_from_cycles(ranges, means, counts,
                             delta_sigma_C,
                             sigma_u,
                             sn_model="eurocode",
                             apply_goodman=True):
    """
    this takes cycles as input and outputs the damage as defined by Palmgren and Miner.
    can also calculate the goodman correction (mean tensile stress possible impact), but regulatory code already takes into account, for most part.
    """
    ranges = np.asarray(ranges, dtype=float)
    means  = np.asarray(means, dtype=float)
    counts = np.asarray(counts, dtype=float)

    if apply_goodman:
        mean_pos = np.maximum(means, 0.0) # only use tensile 
        denom = np.clip(1.0 - mean_pos / sigma_u, 1e-6, None) #clipping works on min value allowed
        ranges_eq = ranges / denom 
    else:
        ranges_eq = ranges

    N_i = cycles_to_failure(ranges_eq, delta_sigma_C, sn_model=sn_model)

    with np.errstate(divide="ignore", invalid="ignore"):
        D_i = np.where(np.isfinite(N_i), counts / N_i, 0.0) #where true, return x, otherwise y

    D = float(np.sum(D_i))

    details = {
        "ranges":     ranges,
        "ranges_eq":  ranges_eq,
        "means":      means,
        "counts":     counts,
        "N_i":        N_i,
        "D_i":        D_i,
    }
    return D, details
